### MIDTERM Assignment 3: Performance Benchmarking and Optimization
**STEP 1**: Set Up the Environment

In [1]:
import time
import multiprocessing
import cProfile

**STEP 2**: Create a Sequential Program

1. Define a function that performs a **compute-intensive task** (e.g., summing squares of numbers).

In [2]:
def compute_sum(data):
  total = 0
  for x in data:
    total += x * x
  return total

2. Generate a large dataset:

In [3]:
data = list(range(1, 10_000_000))

3. Measure execution time of the sequential version:

In [4]:
start_time = time.time()
result = compute_sum(data)
end_time = time.time()
serial_time = end_time - start_time
print("Sequential Execution Time:", serial_time)

Sequential Execution Time: 1.0367918014526367


**STEP 3**: Create a Parallel Version

1. Split the dataset into chunks:

In [5]:
def chunk_data(data, num_chunks):
    chunk_size = len(data) // num_chunks
    return [data[i * chunk_size : (i + 1) * chunk_size] for i in range(num_chunks)]

2. Define a parallel computation using multiprocessing:

In [6]:
def parallel_compute(data):
  num_processes = multiprocessing.cpu_count()
  chunks = chunk_data(data, num_processes)
  with multiprocessing.Pool(processes=num_processes) as pool:
    results = pool.map(compute_sum, chunks)
    return sum(results)

3. Measure execution time of the **parallel version**:

In [7]:
start_time = time.time()
parallel_result = parallel_compute(data)
end_time = time.time()
parallel_time = end_time - start_time
print("Parallel Execution Time:", parallel_time)

Parallel Execution Time: 3.6088781356811523


**STEP 4**: Calculate Speedup and Efficiency

1. Compute **speedup**:

In [8]:
speedup = serial_time / parallel_time
print("Speedup:", speedup)

Speedup: 0.28728922464901935


2. Compute **efficiency**:

In [9]:
num_processors = multiprocessing.cpu_count()
efficiency = speedup / num_processors
print("Efficiency:", efficiency)

Efficiency: 0.14364461232450967


3. Record results in a simple table:

In [10]:
from IPython.display import display, Markdown

table = f"""
| Version | Execution Time (seconds) |
| :--- | :--- |
| **Sequential** | `{serial_time}` |
| **Parallel** | `{parallel_time}` |
"""

display(Markdown(table))


| Version | Execution Time (seconds) |
| :--- | :--- |
| **Sequential** | `1.0367918014526367` |
| **Parallel** | `3.6088781356811523` |


**STEP 5**: Profile the Program (Identify Bottlenecks)

1. Profile the sequential version:

In [11]:
cProfile.run("compute_sum(data)")

         476 function calls (470 primitive calls) in 1.479 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.800    0.800    0.800    0.800 1933115237.py:1(compute_sum)
        3    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
        5    0.000    0.000    0.000    0.000 <frozen importlib._bootstrap>:1390(_handle_fromlist)
        1    0.532    0.532    1.331    1.331 <string>:1(<module>)
        2    0.000    0.000    0.000    0.000 asyncio.py:231(add_callback)
        5    0.000    0.000    0.000    0.000 attrsettr.py:43(__getattr__)
        5    0.000    0.000    0.000    0.000 attrsettr.py:66(_get_attr_opt)
        6    0.000    0.000    1.479    0.246 base_events.py:1922(_run_once)
        2    0.000    0.000    0.000    0.000 base_events.py:2017(get_debug)
        2    0.000    0.000    0.000    0.000 base_events.py:543(_check_closed)
        8    0.000    0.000    0.000    0.000 

2. Observe

1. Which functions take the most time?  
The function `compute_sum` takes the most time, with about 0.800 seconds of execution. Most of the other time shown in the profiler comes from system or Jupyter processes, not from the main code.

2. Where optimization might help?  
Optimization should focus on the `compute_sum` function because it is where most of the program’s time is spent. Improving this function will have the biggest impact on overall performance.

3. Write down **one performance bottleneck** you observe.

One clear performance bottleneck is the use of a standard Python `for` loop inside `compute_sum`, where each element is processed one by one. This approach becomes inefficient when handling very large datasets.

**STEP 6**: Apply One Simple Optimization

**Reduce Loop Overhead**

Replace manual loop with built-in function:

In [12]:
def compute_sum_optimized(data):
  return sum(x*x for x in data)

**STEP 7**: Re-Test Performance

1. Measure execution time again after optimization.

In [13]:
start_time = time.time()
result = compute_sum_optimized(data)
end_time = time.time()
optimized_version_time = end_time - start_time

2. Compare:

* Original sequential
* Parallel
* Optimized Version

In [14]:
print("Original Sequential Execution Time:", serial_time)
print("Original Parallel Execution Time:", parallel_time)
print("Optimized Version:", optimized_version_time)

Original Sequential Execution Time: 1.0367918014526367
Original Parallel Execution Time: 3.6088781356811523
Optimized Version: 0.8110928535461426


1. Which version was fastest?  
The optimized version was the fastest, with an execution time of 0.811 seconds, compared to the sequential version (1.037 seconds) and the parallel version (3.609 seconds). This shows that the optimization had a significant impact on improving performance, outperforming both the original sequential and parallel implementations.

2. What was the speedup achieved?  
The speedup is computed by dividing the sequential execution time by the optimized execution time: 1.0368 ÷ 0.8111 ≈ 1.28. This means the optimized version is about 1.28× faster than the original sequential version, indicating a moderate but meaningful performance improvement.

3. Was efficiency high or low? Why?  
The efficiency was low, particularly for the parallel version, since it performed worse than the sequential version. This low efficiency is mainly due to overhead from multiprocessing, such as process creation, communication, and data splitting, which outweighed the benefits of parallel execution for this specific workload.

4. What bottleneck did profiling reveal?  
Profiling revealed that the main bottleneck was the Python for loop inside the compute_sum function. This loop processes elements one at a time, leading to significant overhead when handling large datasets and limiting overall performance.

5. How did optimization affect performance?  
Optimization improved performance by reducing unnecessary overhead in the computation process. By addressing inefficiencies in the loop and execution flow, the execution time decreased from 1.037 seconds to 0.811 seconds, resulting in faster and more efficient processing overall.